In [ ]:
IS_COLAB = False
# IS_COLAB = True

In [ ]:
# Paths for local and Colab environments

PYTHON_SCRIPT_LOCAL = "main.py" # local
PYTHON_SCRIPT_COLAB = "main.py" # colab

DATA_ROOT_LOCAL = "data/HumanML3D" # local
DATA_ROOT_COLAB = "data/HumanML3D" # colab

CKPT_ROOT_LOCAL = "models/ckpt.pt" # local
CKPT_ROOT_COLAB = "/content/drive/MyDrive/mmdit-motion/models/ckpt.pt" # colab

GRADIO_DEMO_SCRIPT_LOCAL = "gradio-test.py" # local
GRADIO_DEMO_SCRIPT_COLAB = "gradio-test.py" # colab

if IS_COLAB:
    PYTHON_SCRIPT = PYTHON_SCRIPT_COLAB
    DATA_ROOT = DATA_ROOT_COLAB
    CKPT_ROOT = CKPT_ROOT_COLAB
    GRADIO_DEMO_SCRIPT = GRADIO_DEMO_SCRIPT_COLAB
else:
    PYTHON_SCRIPT = PYTHON_SCRIPT_LOCAL
    DATA_ROOT = DATA_ROOT_LOCAL
    CKPT_ROOT = CKPT_ROOT_LOCAL
    GRADIO_DEMO_SCRIPT = GRADIO_DEMO_SCRIPT_LOCAL

In [ ]:
# If using Colab, mount Google Drive and install dependencies

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install torchinfo gradio transformers datasets mmdit ema-pytorch

    !nvidia-smi --query-gpu=name --format=csv,noheader

    !rm -rf mmdit-test
    !git clone https://github.com/chihhsiangfu/mmdit-test
    !cp -r mmdit-test/mmdit-motion/main.py main.py
    !cp -r mmdit-test/mmdit-motion/gradio-test.py gradio-test.py

In [ ]:
# Download the dataset

!python $PYTHON_SCRIPT download --data_root $DATA_ROOT

In [ ]:
# Train the model

!python $PYTHON_SCRIPT train \
    --data_root $DATA_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --batch_size 64 \
    --steps 200000 \
    --ckpt_out $CKPT_ROOT

# Terminate Colab runtime to free up resources after training

if IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
# Sample motion from the trained model

!python $PYTHON_SCRIPT sample \
    --resume $CKPT_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --prompt "a person walks forward then sits down" \
    --steps 50 --cfg 4.0 \
    --out output/sample.npy

In [ ]:
# Run the Gradio demo

SHARE = "--share" if IS_COLAB else ""

!python $GRADIO_DEMO_SCRIPT \
    --ckpt $CKPT_ROOT \
    --use_ema \
    $SHARE